In [12]:
import requests
from IPython.display import display, HTML

In [15]:
from bs4 import BeautifulSoup
import sys 
sys.path.insert(0, '../scripts-homepage-data/')
import get_bounding_boxes_from_html as bb

In [13]:
video_url = 'https://sanfrancisco.granicus.com/player/clip/43243?view_id=10&meta_id=992444&redirect=true&h=b9650faa96d53d07034d556d92f18771'

In [14]:
resp = requests.get(video_url)
html = resp.text

In [16]:
soup = BeautifulSoup(html, 'lxml')

In [58]:
soup.find('section', {'id': 'index'})

<section aria-labelledby="index-section-label" class="index-content index-content-visible" id="index">
<div class="index-point flex-col-center" data-id="992403" data-uid="7709eebe-ad11-4833-9291-81e749ef1520" time="44">
                    1 ROLL CALL AND PLEDGE OF ALLEGIANCE
                </div>
<div class="index-point flex-col-center" data-id="992404" data-uid="6e61eb48-161c-4606-b208-12c327af7d37" time="118">
                    2 COMMUNICATIONS
                </div>
<div class="index-point flex-col-center" data-id="992405" data-uid="cfae1419-7455-4198-8a91-9a3e3c3b9c00" time="163">
                    3 APPROVAL OF MEETING MINUTES
                </div>
<div class="index-point flex-col-center" data-id="992408" data-uid="e2e31e34-68be-450c-a69c-d4a248a54536" time="226">
                    4 CONSENT AGENDA[Consent Boiler|B1]
                </div>
<div class="index-point flex-col-center" data-id="992419" data-uid="92ad5a30-4c86-409b-bec0-660b6e42fe5e" time="279">
                

# Download Audio

In [17]:
download_links = soup.find('div', {'id': 'download-options'}).find_all('a')

In [18]:
audio_link = list(filter(lambda x: 'audio' in x.get_text().strip().lower(), download_links))[0]
audio_download_link = audio_link.get('href')

In [20]:
audio_file = requests.get(audio_download_link).content

In [25]:
filename = audio_download_link.split('/')[-1]

In [30]:
file_outpath = 'cache/' + filename
with open(file_outpath, 'wb') as f:
    f.write(audio_file)

In [28]:
import whisper

model = whisper.load_model("base.en")

100%|███████████████████████████████████████| 139M/139M [00:48<00:00, 3.02MiB/s]


In [ ]:
# load audio and pad/trim it to fit 30 seconds
audio = whisper.load_audio(file_outpath)
audio = whisper.pad_or_trim(audio)

# make log-Mel spectrogram and move to the same device as the model
mel = whisper.log_mel_spectrogram(audio).to(model.device)

In [ ]:
# decode the audio
options = whisper.DecodingOptions()
result = whisper.decode(model, mel, options)

In [ ]:
# print the recognized text
print(result.text)

In [33]:
audio_download_link

'https://archive-video.granicus.com/sanfrancisco/sanfrancisco_05be178e-c8c6-11ed-acbc-0050569183fa.mp3'

# Get Playwright to Get to the Video Page

In [34]:
from bs4 import BeautifulSoup

In [35]:
soup = BeautifulSoup('<a id="ctl00_ContentPlaceHolder1_gridCalendar_ctl00_ctl70_hypVideo" class="videoFileNotAvailableLink videolink" data-event-id="92160b91-a4cf-4db5-af96-91606842a590" data-running-text="In&amp;nbsp;progress" style="color:Gray;font-family:Tahoma;font-size:10pt;">Not&nbsp;available</a>')

In [41]:
soup.find('a').get_attribute_list('onclick')

[None]